In [2]:
!pip install python-dotenv

In [27]:
import os
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict
from getpass import getpass
from typing import Dict, Any

In [ ]:


env_path = Path("/Users/helen/hdres/AI agent/.env")

api_key = getpass("Paste your REAL API key: ")

env_content = f'''LLM_API_KEY="{api_key}"
LLM_MODEL_ID="openai-gpt-55-pro"
LLM_BASE_URL="https://api.venice.ai/api/v1"
'''

env_path.write_text(env_content, encoding="utf-8")

print("Fixed .env:", env_path)

Fixed .env: /Users/helen/hdres/AI agent/.env


In [ ]:
env_path = Path("/Users/helen/hdres/AI agent/.env")
load_dotenv(env_path, override=True)

print("Env exists:", env_path.exists())
print("LLM_API_KEY loaded:", os.getenv("LLM_API_KEY") is not None)
print("LLM_MODEL_ID:", os.getenv("LLM_MODEL_ID"))
print("LLM_BASE_URL:", os.getenv("LLM_BASE_URL"))

Env exists: True
LLM_API_KEY loaded: True
LLM_MODEL_ID: openai-gpt-55-pro
LLM_BASE_URL: https://api.venice.ai/api/v1


In [ ]:
class HelloAgentsLLM:
    """
    A customized LLM client for the book "Hello Agents".
    It is used to call any service compatible with the OpenAI interface and uses streaming responses by default.
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        Initialize the client. Prioritize passed parameters; if not provided, load from environment variables.
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("Model ID, API key, and service address must be provided or defined in the .env file.")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        Call the large language model to think and return its response.
        """
        print(f"🧠 Calling {self.model} model...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # Handle streaming response
            print("✅ Large language model response successful.")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                collected_content.append(content)
            print()  # Newline after streaming output ends
            return "".join(collected_content).strip()

        except Exception as e:
            print(f"❌ Error occurred when calling LLM API: {e}")
            return None


In [55]:
pip install google-search-results


Note: you may need to restart the kernel to use updated packages.


In [5]:
env_path = Path("/Users/helen/hdres/AI agent/.env")

# Make sure .env exists
if not env_path.exists():
    env_path.write_text("", encoding="utf-8")

# Read existing .env content
existing_content = env_path.read_text(encoding="utf-8")

# Ask for SerpApi key safely
serpapi_key = getpass("Paste your real SERPAPI_API_KEY: ")

# Remove old SERPAPI_API_KEY line if it already exists
lines = existing_content.splitlines()
lines = [line for line in lines if not line.startswith("SERPAPI_API_KEY=")]

# Add new SERPAPI_API_KEY
lines.append(f'SERPAPI_API_KEY="{serpapi_key}"')

# Write back to .env
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("SERPAPI_API_KEY saved to:", env_path)

SERPAPI_API_KEY saved to: /Users/helen/hdres/AI agent/.env


In [15]:
env_path = Path("/Users/helen/hdres/AI agent/.env")

print("Env exists:", env_path.exists())
print("Env path:", env_path)

print("\n--- .env content, keys hidden ---")
for line in env_path.read_text(encoding="utf-8").splitlines():
    if line.startswith("LLM_API_KEY="):
        print("LLM_API_KEY=***hidden***")
    elif line.startswith("SERPAPI_API_KEY="):
        print("SERPAPI_API_KEY=***hidden***")
    else:
        print(line)

Env exists: True
Env path: /Users/helen/hdres/AI agent/.env

--- .env content, keys hidden ---
LLM_API_KEY=***hidden***
LLM_MODEL_ID="openai-gpt-55-pro"
LLM_BASE_URL="https://api.venice.ai/api/v1"
SERPAPI_API_KEY=***hidden***


In [17]:
env_path = Path("/Users/helen/hdres/AI agent/.env")
load_dotenv(env_path, override=True)

print("LLM_API_KEY loaded:", os.getenv("LLM_API_KEY") is not None)
print("LLM_MODEL_ID:", os.getenv("LLM_MODEL_ID"))
print("LLM_BASE_URL:", os.getenv("LLM_BASE_URL"))
print("SERPAPI_API_KEY loaded:", os.getenv("SERPAPI_API_KEY") is not None)

LLM_API_KEY loaded: True
LLM_MODEL_ID: openai-gpt-55-pro
LLM_BASE_URL: https://api.venice.ai/api/v1
SERPAPI_API_KEY loaded: True


In [18]:
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    A practical web search engine tool based on SerpApi.
    It intelligently parses search results, prioritizing direct answers or knowledge graph information.
    """
    print(f"🔍 Executing [SerpApi] web search: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "Error: SERPAPI_API_KEY not configured in .env file."

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # Country code
            "hl": "zh-cn", # Language code
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # Intelligent parsing: prioritize finding the most direct answer
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # If no direct answer, return summaries of the first three organic results
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"Sorry, no information found about '{query}'."

    except Exception as e:
        return f"Error occurred during search: {e}"


In [ ]:


class ToolExecutor:
    """
    A tool executor responsible for managing and executing tools.
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        Register a new tool in the toolbox.
        """
        if name in self.tools:
            print(f"Warning: Tool '{name}' already exists and will be overwritten.")
        self.tools[name] = {"description": description, "func": func}
        print(f"Tool '{name}' registered.")

    def getTool(self, name: str) -> callable:
        """
        Get a tool's execution function by name.
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        Get a formatted description string of all available tools.
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])


In [ ]:
REACT_PROMPT_TEMPLATE = """
You are a ReAct agent. You can reason and call external tools.

Available tools:
{tools}

You must follow these rules:

1. If the question asks about latest, current, recent, today, newest, or any fact that may change over time, you MUST call Search before giving the final answer.
2. If the question asks about a current product, release, price, date, ranking, news, or recent event, you MUST NOT answer from memory.
3. If History is empty and the question requires current information, your first Action must be Search[search query].
4. After you receive an Observation and have enough information, then use Finish[final answer].
5. Output exactly two fields and nothing else:
Thought: your reasoning
Action: Search[query] or Finish[final answer]

Question: {question}

History:
{history}
"""

In [24]:
class ReActAgent:
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        Run the ReAct agent to answer a question.
        """
        self.history = []
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- Step {current_step} ---")

            # 1. Format prompt
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)

            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. Call LLM
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)

            if not response_text:
                print("Error: LLM failed to return a valid response.")
                break

            # 3. Parse LLM output
            thought, action = self._parse_output(response_text)

            if thought:
                print(f"Thought: {thought}")

            if not action:
                print("Warning: Failed to parse valid Action, process terminated.")
                break

            # 4. Execute Action
            if action.startswith("Finish"):
                final_match = re.match(r"Finish\[(.*)\]", action, re.DOTALL)

                if final_match:
                    final_answer = final_match.group(1).strip()
                    print(f"🎉 Final Answer: {final_answer}")
                    return final_answer
                else:
                    print("Warning: Failed to parse Finish action.")
                    break

            tool_name, tool_input = self._parse_action(action)

            if not tool_name or not tool_input:
                print(f"Warning: Invalid action format: {action}")
                continue

            print(f"🎬 Action: {tool_name}[{tool_input}]")

            tool_function = self.tool_executor.getTool(tool_name)

            if not tool_function:
                observation = f"Error: Tool named '{tool_name}' not found."
            else:
                observation = tool_function(tool_input)

            print(f"👀 Observation: {observation}")

            # 5. Add this round's Thought, Action, and Observation to history
            self.history.append(f"Thought: {thought}")
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        print("Maximum steps reached, process terminated.")
        return None

    def _parse_output(self, text: str):
        """
        Parse LLM output to extract Thought and Action.
        """
        thought_match = re.search(
            r"Thought:\s*(.*?)(?=\n\s*Action:|$)",
            text,
            re.DOTALL
        )

        action_match = re.search(
            r"Action:\s*([A-Za-z_]\w*\[.*\])",
            text,
            re.DOTALL
        )

        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None

        return thought, action

    def _parse_action(self, action_text: str):
        """
        Parse Action string to extract tool name and input.
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)

        if match:
            tool_name = match.group(1).strip()
            tool_input = match.group(2).strip()
            return tool_name, tool_input

        return None, None

In [26]:
llm = HelloAgentsLLM()

tool_executor = ToolExecutor()

search_desc = (
    "A web search engine. Use this tool when you need to answer questions "
    "about current events, facts, and information not found in your knowledge base."
)

tool_executor.registerTool("Search", search_desc, search)

agent = ReActAgent(
    llm_client=llm,
    tool_executor=tool_executor,
    max_steps=5
)

question = "What is the model name of Huawei's latest mobile phone? What are its selling points?"

agent.run(question)

Tool 'Search' registered.
--- Step 1 ---
🧠 Calling openai-gpt-55-pro model...
✅ Large language model response successful.

Thought: I need up-to-date information about Huawei’s newest phone model and its main selling points, so I should check current web information.
🎬 Action: Search[Huawei latest mobile phone model 2026 selling points]
🔍 Executing [SerpApi] web search: Huawei latest mobile phone model 2026 selling points
👀 Observation: [1] HUAWEI Phones
Explore the latest technologies in smartphone with HUAWEI, and check out the HUAWEI Mate series, HUAWEI Pura series and HUAWEI nova series.

[2] The best Huawei phones in 2026
The latest and greatest is the new and fresh Pura 80 Ultra, with its one inch sensor and a very interesting design. Let's not forget the first ...

[3] Huawei Phones and Prices Guide 2026 — How to Choose ...
A practical, data-driven Huawei phones and prices guide for 2026. Learn which models offer real value before the July 1 price hike—and when ...
--- Step 2 --

'Huawei’s latest flagship mobile phone is the HUAWEI Pura 80 Ultra.\n\nIts main selling points are:\n1. Camera-first flagship: a large one-inch-class main camera sensor focused on high detail, low-light performance, and strong dynamic range.\n2. Advanced zoom photography: a high-end telephoto/zoom system designed for portraits, distant subjects, and clearer long-range shots.\n3. Premium design: distinctive Pura-series styling with a luxury flagship look.\n4. High-end display: large OLED screen with smooth refresh rate and flagship-level brightness/color performance.\n5. Fast charging and strong battery life: supports Huawei’s high-speed wired and wireless charging on the Ultra model.\n6. Durability: premium glass/body construction with strong water and dust resistance on flagship variants.\n7. Huawei ecosystem features: HarmonyOS/EMUI features, Huawei device integration, AI photo tools, and—in some markets—satellite communication features.\n\nIn short: the Pura 80 Ultra is positioned m